# U-Net Segmentation Training - Google Colab Orchestrator

**Orchestrator only.** All training logic lives in the main repository files
(`train.py`, `utils/data_loading.py`, `unet/`, etc.). This notebook simply clones
the repo, installs dependencies, and calls into the existing code.

---

| Item | Value |
|------|-------|
| Source repo | `https://github.com/HaikalFK/segmentasi-unet.git` |
| Dataset | Plant Phenotyping (Kaggle) |
| Classes | 22 (auto-detected) |
| Runtime | **GPU** (T4, V100, or A100) |

---
## 1. Install Dependencies

PyTorch with CUDA is pre-installed in Colab. We only need to install
the project-specific dependencies from `requirements.txt`.

In [1]:
# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU device: Tesla T4


In [2]:
# Clone the repository (or pull latest if already cloned)
import os
from pathlib import Path

REPO_URL = 'https://github.com/HaikalFK/segmentasi-unet.git'
REPO_DIR = Path('/content/segmentasi-unet')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Fetching latest...')
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/main

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

Cloning repository...
Cloning into '/content/segmentasi-unet'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 111 (delta 34), reused 99 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 86.30 KiB | 1.18 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/segmentasi-unet
Working directory: /content/segmentasi-unet


In [3]:
# Clone the repository (or pull latest if already cloned)
import os
from pathlib import Path

REPO_URL = 'https://github.com/HaikalFK/segmentasi-unet.git'
REPO_DIR = Path('/content/segmentasi-unet')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Fetching latest...')
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/main

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

Repository already cloned. Fetching latest...
/content/segmentasi-unet
Fetching origin
HEAD is now at 7ba20f7 Merge pull request #6 from HaikalFK/train_config
/content/segmentasi-unet
Working directory: /content/segmentasi-unet


# Install project dependencies (version ranges to avoid build failures on Colab)
# PyTorch with CUDA is already pre-installed in Colab.
!pip install --quiet --upgrade pip
!pip install --quiet -r requirements.txt

# Verify imports
from utils.data_loading import BasicDataset
from unet import UNet
print('All imports OK')

In [4]:
!python data/download_dataset.py

Plant Phenotyping Dataset Downloader

Downloading... (this may take a while for large datasets)
Using Colab cache for faster access to the 'plant-phenotyping-dataset' dataset.
Dataset downloaded to: /kaggle/input/plant-phenotyping-dataset
Dataset root: /kaggle/input/plant-phenotyping-dataset/Plant_Phenotyping_Datasets

Organizing dataset into data/imgs/ and data/masks/

  Processing: Plant/Ara2012
    Found 120 RGB images
    Copied: 120 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Ara2013-Canon
    Found 165 RGB images
    Copied: 165 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Tobacco
    Found 62 RGB images
    Copied: 62 image/mask pairs, Skipped: 0 (no label)

  Total: 347 image/mask pairs copied

Verifying dataset
  Images: 347
  Masks:  347
  Matched pairs: 347

Done! Dataset is ready for training.
  Images: /content/segmentasi-unet/data/imgs  (347 files)
  Masks:  /content/segmentasi-unet/data/masks  (347 files)
  Matched pairs: 347

To train th

---
## 3. Run Training

All parameters are passed as CLI arguments to `train.py`.
Edit the variables below to configure training.

In [5]:
# ============================================================
# TRAINING CONFIGURATION
# Edit these values as needed.
# ============================================================
EPOCHS        = 100
BATCH_SIZE    = 8
SCALE         = 0.5
LEARNING_RATE = 1e-5
VALIDATION    = 10.0
AMP           = True
BILINEAR      = False

print('Configuration:')
print(f'  Epochs:    {EPOCHS}')
print(f'  Batch:     {BATCH_SIZE}')
print(f'  Scale:     {SCALE}')
print(f'  AMP:       {AMP}')
print(f'  Bilinear:  {BILINEAR}')

Configuration:
  Epochs:    100
  Batch:     8
  Scale:     0.5
  AMP:       True
  Bilinear:  False


In [6]:
# Build and execute the training command
cmd = (
    f'python train.py'
    f' --classes 21'
    f' --epochs {EPOCHS}'
    f' --batch-size {BATCH_SIZE}'
    f' --scale {SCALE}'
    f' --learning-rate {LEARNING_RATE}'
    f' --validation {VALIDATION}'
)

if AMP:
    cmd += ' --amp'
if BILINEAR:
    cmd += ' --bilinear'

print(f'Command: {cmd}')
print('=' * 70)
!{cmd}

Command: python train.py --classes 21 --epochs 100 --batch-size 8 --scale 0.5 --learning-rate 1e-05 --validation 10.0 --amp
INFO: Using device cuda
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:06<00:00, 53.68it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
INFO: Detected 22 classes from mask files (override --classes 21)
INFO: Network:
	3 input channels
	22 output channels (classes)
	Transposed conv upscaling
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
  0% 0/347 [00:00<?, ?it/s]
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:06<00:00, 51.55it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
]11;?

# ============================================================
# TRAINING CONFIGURATION
# Edit these values as needed. All arguments are passed
# directly to train.py from the main repo.
# ============================================================

In [7]:
EPOCHS              = 100
BATCH_SIZE          = 8
SCALE               = 0.5
LEARNING_RATE       = 1e-5
VALIDATION          = 10.0
AMP                 = True
BILINEAR            = False
EARLY_STOP_PATIENCE = 15
EARLY_STOP_DELTA    = 0.001

print('Configuration:')
print(f'  Epochs:              {EPOCHS}')
print(f'  Batch:               {BATCH_SIZE}')
print(f'  Scale:               {SCALE}')
print(f'  AMP:                 {AMP}')
print(f'  Bilinear:            {BILINEAR}')
print(f'  Early stop patience: {EARLY_STOP_PATIENCE}')
print(f'  Early stop delta:    {EARLY_STOP_DELTA}')

Configuration:
  Epochs:              100
  Batch:               8
  Scale:               0.5
  AMP:                 True
  Bilinear:            False
  Early stop patience: 15
  Early stop delta:    0.001


In [8]:
from pathlib import Path
checkpoints = sorted(Path('checkpoints').glob('*.pth'))
print(f'Checkpoints found: {len(checkpoints)}')
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / (1024 * 1024)
    print(f'  {ckpt.name}  ({size_mb:.2f} MB)')

Checkpoints found: 0


In [9]:
# Build and execute the training command
cmd = (
    f'python train.py'
    f' --classes 21'
    f' --epochs {EPOCHS}'
    f' --batch-size {BATCH_SIZE}'
    f' --scale {SCALE}'
    f' --learning-rate {LEARNING_RATE}'
    f' --validation {VALIDATION}'
    f' --early-stop-patience {EARLY_STOP_PATIENCE}'
    f' --early-stop-delta {EARLY_STOP_DELTA}'
)

if AMP:
    cmd += ' --amp'
if BILINEAR:
    cmd += ' --bilinear'

print(f'Command: {cmd}')
print('=' * 70)
!{cmd}

Command: python train.py --classes 21 --epochs 100 --batch-size 8 --scale 0.5 --learning-rate 1e-05 --validation 10.0 --early-stop-patience 15 --early-stop-delta 0.001 --amp
INFO: Using device cuda
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:07<00:00, 47.89it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
INFO: Detected 22 classes from mask files (override --classes 21)
INFO: Network:
	3 input channels
	22 output channels (classes)
	Transposed conv upscaling
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
  0% 0/347 [00:00<?, ?it/s]
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:06<00:00, 57.73it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
wandb: WARNING The anonymous setting has no effect

---
## 5. Evaluate Model

Run evaluation using `evaluate.py` against a trained checkpoint.

In [10]:
CHECKPOINT_PATH = 'checkpoints/checkpoint_epoch100.pth'  # adjust
!python evaluate.py --load {CHECKPOINT_PATH} --classes 21 --scale {SCALE}

---
## 6. Predict on a Sample Image

Uses `predict.py` from the main repo.

In [11]:
CHECKPOINT_PATH = 'checkpoints/checkpoint_epoch3.pth'  # adjust
SAMPLE_IMAGE = 'data/imgs/ara2012_plant001.png'

!python predict.py --model {CHECKPOINT_PATH} --input {SAMPLE_IMAGE} --classes 21 --scale {SCALE} --viz

INFO: Loading model checkpoints/checkpoint_epoch3.pth
INFO: Using device cuda
Traceback (most recent call last):
  File "/content/segmentasi-unet/predict.py", line 93, in <module>
    state_dict = torch.load(args.model, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/checkpoint_epoch3.pth'
